# forecast_result_check

`run_trade_adv_forecast_v2_part1/2/3.py` 로 DB(`korea_monthly_trade_forecast_v2`)에 저장한 예측 결과가 **제대로 들어갔는지 계량**하고, 가장 최근 forecast_date의 예측치(원칙상 24개월)를 CSV로 저장하는 노트북입니다.

1. 최근 forecast_date 자동 확인
2. HS코드별 예측 개월수 계량 (24개월 기준 미달/초과 탐지)
3. 최근 예측분(24개월) 전체를 `test_result_{날짜}.csv` 로 저장


In [10]:
import os
import pymysql
import pandas as pd

from trade_data_import import get_trade_data_by_hscode, get_unique_hscode_list
from DATA.stock_invest_function import *

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

FORECAST_TABLE = 'korea_monthly_trade_forecast_v2'
INDICATOR = 'ensemble_expDlr'   # run_trade_adv_forecast_v2_*.py 가 저장하는 앙상블 예측 indicator
EXPECTED_HORIZON = 24           # 원본 스크립트의 horizon=24 기준


## 1. 조회 함수
기존 `get_unique_hscode_by_forecast_date`는 그대로 두고, 실제 예측 값(long format: hs_code, date, value)을 가져오는 함수와 forecast_date 목록을 가져오는 함수를 추가합니다.

In [11]:
def get_unique_hscode_by_forecast_date(db_info: dict, forecast_date: str):
    """
    주어진 forecast_date 에 대해 unique hs_code 값을 반환.
    """
    query = """
        SELECT DISTINCT hs_code
        FROM korea_monthly_trade_forecast_v2
        WHERE forecast_date = %s
        ORDER BY hs_code;
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        df = pd.read_sql(query, conn, params=[forecast_date])
        return df["hs_code"].tolist()

    finally:
        conn.close()


def get_unique_forecast_dates(db_info: dict, table_name: str = FORECAST_TABLE):
    """
    forecast_forecast_v2 테이블에 존재하는 forecast_date 전체 목록(오름차순)을 반환.
    가장 최근 배치가 언제 들어갔는지 확인할 때 사용.
    """
    query = f"""
        SELECT DISTINCT forecast_date
        FROM {table_name}
        ORDER BY forecast_date;
    """
    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"], user=db_info["user"],
        password=db_info["password"], database=db_info["database"], charset="utf8mb4"
    )
    try:
        df = pd.read_sql(query, conn)
        return pd.to_datetime(df["forecast_date"]).dt.strftime("%Y-%m-%d").tolist()
    finally:
        conn.close()


def get_forecast_long(db_info: dict, forecast_date: str, indicator: str = INDICATOR,
                       table_name: str = FORECAST_TABLE) -> pd.DataFrame:
    """
    특정 forecast_date + indicator 로 저장된 예측치를 long format(hs_code, date, value)으로 전부 가져온다.
    run_trade_adv_forecast_v2_*.py 가 DB에 실제로 뭘 저장했는지를 그대로 보는 함수.
    """
    query = f"""
        SELECT hs_code, date, value
        FROM {table_name}
        WHERE forecast_date = %s
          AND indicator = %s
        ORDER BY hs_code, date;
    """
    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"], user=db_info["user"],
        password=db_info["password"], database=db_info["database"], charset="utf8mb4"
    )
    try:
        df = pd.read_sql(query, conn, params=[forecast_date, indicator])
        df["date"] = pd.to_datetime(df["date"])
        df["hs_code"] = df["hs_code"].astype(str)
        return df
    finally:
        conn.close()


## 2. 가장 최근 forecast_date 확인
`run_trade_adv_forecast_v2_part1/2/3.py` 를 오늘 돌렸다면, 이 값이 오늘 날짜(혹은 실행 시점)로 찍혀야 정상입니다.

In [14]:
all_forecast_dates = get_unique_forecast_dates(db_info)
LATEST_FORECAST_DATE = all_forecast_dates[-1]

print("저장된 forecast_date 개수:", len(all_forecast_dates))
print("가장 최근 5개:", all_forecast_dates[-5:])
print("-> 검증에 사용할 최근 forecast_date:", LATEST_FORECAST_DATE)


저장된 forecast_date 개수: 26
가장 최근 5개: ['2026-08-13', '2026-08-14', '2026-08-15', '2026-09-15', '2026-09-18']
-> 검증에 사용할 최근 forecast_date: 2026-09-18


## 3. HS코드별 예측 개월수 계량
이번 배치(`LATEST_FORECAST_DATE`)로 들어간 hs_code별 예측 데이터가 정말 24개월인지, 부족한 코드는 몇 개나 되는지 계량합니다.

In [15]:
forecast_long = get_forecast_long(db_info, LATEST_FORECAST_DATE, indicator=INDICATOR)

print(f"forecast_date={LATEST_FORECAST_DATE} / indicator={INDICATOR}")
print("총 행수:", len(forecast_long))
print("고유 hs_code 개수:", forecast_long['hs_code'].nunique())
forecast_long.head()


forecast_date=2026-09-18 / indicator=ensemble_expDlr
총 행수: 11256
고유 hs_code 개수: 469


,hs_code,date,value
0,030354,2026-09-30,2.372583e+07
1,030354,2026-10-31,2.332668e+07
2,030354,2026-11-30,2.649436e+07
3,030354,2026-12-31,3.445929e+07
4,030354,2027-01-31,2.841546e+07


In [8]:
# hs_code 별 개월수 / 기간 계량
summary = (
    forecast_long
    .groupby("hs_code")["date"]
    .agg(개월수="count", 시작월="min", 종료월="max")
    .reset_index()
)
summary["기대개월수"] = EXPECTED_HORIZON
summary["차이"] = summary["기대개월수"] - summary["개월수"]
summary = summary.sort_values("차이", ascending=False)

n_ok = (summary["차이"] == 0).sum()
n_bad = (summary["차이"] != 0).sum()

print(f"[계량 결과] forecast_date={LATEST_FORECAST_DATE}")
print(f"  - 정확히 {EXPECTED_HORIZON}개월로 들어간 HS코드: {n_ok}개")
print(f"  - {EXPECTED_HORIZON}개월이 아닌 HS코드: {n_bad}개")
if n_bad > 0:
    print("\n  [미달/초과 상세] (상위 20개)")
    display(summary[summary["차이"] != 0].head(20))
else:
    print("  -> 모든 HS코드가 정확히 24개월씩 저장되어 있습니다.")


[계량 결과] forecast_date=2026-09-15
  - 정확히 24개월로 들어간 HS코드: 469개
  - 24개월이 아닌 HS코드: 0개
  -> 모든 HS코드가 정확히 24개월씩 저장되어 있습니다.


## 4. 최근 예측(24개월) 전체를 CSV로 저장
`LATEST_FORECAST_DATE` 기준 전체 hs_code의 예측치를 지정 경로에 저장합니다.
파일명: `test_result_{forecast_date}.csv` (예: forecast_date가 2026-09-18이면 `test_result_20260918.csv`)

In [9]:
SAVE_DIR = r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\한국_월간수출분석\test_result"
os.makedirs(SAVE_DIR, exist_ok=True)

date_str = pd.to_datetime(LATEST_FORECAST_DATE).strftime("%Y%m%d")
save_path = os.path.join(SAVE_DIR, f"test_result_{date_str}.csv")

# long format(hs_code, date, value) 그대로 저장 — DB에 실제 들어간 값을 그대로 검증할 수 있는 형태
export_df = forecast_long.sort_values(["hs_code", "date"]).reset_index(drop=True)
export_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"[저장 완료] {save_path}")
print(f"  - 행수: {len(export_df)}  |  hs_code 수: {export_df['hs_code'].nunique()}")
print(f"  - 기간: {export_df['date'].min().strftime('%Y-%m')} ~ {export_df['date'].max().strftime('%Y-%m')}")


[저장 완료] C:\Users\82108\OneDrive\INVESTMENT\한국주식\한국_월간수출분석\test_result\test_result_20260915.csv
  - 행수: 11256  |  hs_code 수: 469
  - 기간: 2026-08 ~ 2028-08


### (선택) wide 포맷으로도 저장하고 싶다면
이후 다른 분석 노트북들(`trade_total_YYYYMM.csv` 형태)과 바로 이어쓰고 싶으면, 아래처럼 date × hs_code 피벗본을 추가로 저장할 수 있습니다.

In [ ]:
# wide_df = export_df.pivot(index="date", columns="hs_code", values="value").sort_index()
# wide_path = os.path.join(SAVE_DIR, f"test_result_wide_{date_str}.csv")
# wide_df.to_csv(wide_path, encoding="utf-8-sig")
# print(f"[wide 저장 완료] {wide_path}")


## 5. forecast_date별 HS코드 커버리지 확인
DB에는 여러 forecast_date가 쌓여 있을 수 있습니다(예: 오늘 `--range 0:2`로 테스트한 것도 새 forecast_date로 들어갔을 것입니다). **전체 HS코드가 들어있는 '진짜 배치'가 어느 forecast_date인지부터** 확인합니다.

In [ ]:
coverage = []
for fd in all_forecast_dates:
    codes = get_unique_hscode_by_forecast_date(db_info, fd)
    coverage.append({"forecast_date": fd, "hs_code_수": len(codes)})

coverage_df = pd.DataFrame(coverage).sort_values("forecast_date")
print("forecast_date별 hs_code 개수:")
display(coverage_df)


## 6. 전체 HS코드 예측 개월수 계량 (가장 큰 배치 기준)
위에서 hs_code 수가 가장 많은 forecast_date를 '전체 배치'로 간주하고, 거기 속한 **모든 HS코드**의 예측 개월수·종료월을 계량합니다. (테스트용 소규모 배치가 최댓값으로 잘못 잡히지 않도록, 필요하면 `FULL_FORECAST_DATE`를 직접 지정하세요.)

In [ ]:
# 자동 추정: hs_code 수가 가장 많은 forecast_date
FULL_FORECAST_DATE = coverage_df.loc[coverage_df["hs_code_수"].idxmax(), "forecast_date"]

# 자동 추정이 틀렸다면 아래 줄의 주석을 풀고 직접 지정하세요.
# FULL_FORECAST_DATE = "2026-08-13"

print("전체 배치로 사용할 forecast_date:", FULL_FORECAST_DATE)

full_long = get_forecast_long(db_info, FULL_FORECAST_DATE, indicator=INDICATOR)

full_summary = (
    full_long.groupby("hs_code")["date"]
    .agg(개월수="count", 시작월="min", 종료월="max")
    .reset_index()
)
full_summary["기대개월수"] = EXPECTED_HORIZON
full_summary["차이"] = full_summary["기대개월수"] - full_summary["개월수"]
full_summary = full_summary.sort_values("차이", ascending=False).reset_index(drop=True)

n_ok = (full_summary["차이"] == 0).sum()
n_bad = (full_summary["차이"] != 0).sum()

print(f"\n[전체 계량] forecast_date={FULL_FORECAST_DATE}  |  전체 HS코드 수: {len(full_summary)}")
print(f"  - 24개월 정확: {n_ok}개")
print(f"  - 24개월 아님: {n_bad}개")
print()
print("종료월 분포 (몇 개 코드가 어느 달까지 예측되어 있는지):")
display(full_summary["종료월"].value_counts().sort_index())


## 7. 전체 계량 요약을 CSV로 저장 (검토용)
원본 24개월치 값이 아니라, HS코드별 **개월수·시작월·종료월 요약**만 저장합니다. 이 파일을 그대로 업로드하시면 전체 HS코드가 몇 개월씩 들어있는지 한눈에 검토할 수 있습니다.

In [ ]:
summary_date_str = pd.to_datetime(FULL_FORECAST_DATE).strftime("%Y%m%d")
summary_path = os.path.join(SAVE_DIR, f"forecast_horizon_summary_{summary_date_str}.csv")

full_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"[요약 저장 완료] {summary_path}")
print(f"  - 전체 {len(full_summary)}개 HS코드의 개월수/시작월/종료월 요약본입니다.")
